# API v2 - Azure Endpoint Predictions
This notebook sends data to the deployed Azure endpoint and returns model predictions.

In [4]:
import json
import pandas as pd
import requests

REQUIRED_COLUMNS = [
    "StandardCost",
    "ListPrice",
    "Weight",
    "ProductCategoryID",
    "ProductModelID",
]

data = pd.read_csv("prediction_instances.csv")

missing = [c for c in REQUIRED_COLUMNS if c not in data.columns]
if missing:
    raise ValueError(f"Missing required columns in input CSV: {missing}")

data = data[REQUIRED_COLUMNS].copy()
data.head()

,StandardCost,ListPrice,Weight,ProductCategoryID,ProductModelID
0,12.50,24.99,0.45,1,101
1,18.20,34.50,0.62,1,102
2,25.00,49.99,0.80,2,103
3,31.75,59.95,1.05,2,104
4,14.10,27.80,0.50,3,105


In [5]:
with open("uri.json", "r") as f:
    scoring_uri = json.load(f)["URI"][0]

payload = {
    "data": [data.to_dict(orient="list")]
}

headers = {"Content-Type": "application/json"}
response = requests.post(
    scoring_uri,
    headers=headers,
    data=json.dumps(payload),
    timeout=60,
)

print("Status code:", response.status_code)
print("Raw response:", response.text)

Status code: 200
Raw response: "[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"


In [6]:
if response.status_code != 200:
    raise RuntimeError(f"Request failed ({response.status_code}): {response.text}")

body = response.json()

while isinstance(body, str):
    body = json.loads(body)

if isinstance(body, dict):
    if "error" in body:
        raise RuntimeError(f"Endpoint error: {body['error']}")
    if "predictions" in body:
        predictions = body["predictions"]
    else:
        raise RuntimeError(f"Unexpected response shape: {body}")

elif isinstance(body, list):
    predictions = body

else:
    raise RuntimeError(f"Unexpected response type: {type(body).__name__}")

if len(predictions) != len(data):
    raise RuntimeError(
        f"Prediction count mismatch. Got {len(predictions)} for {len(data)} rows."
)

result = data.copy()
result["Prediction"] = predictions
result

,StandardCost,ListPrice,Weight,ProductCategoryID,ProductModelID,Prediction
0,12.50,24.99,0.45,1,101,1
1,18.20,34.50,0.62,1,102,1
2,25.00,49.99,0.80,2,103,1
3,31.75,59.95,1.05,2,104,1
4,14.10,27.80,0.50,3,105,1
5,42.00,79.90,1.40,3,106,1
6,9.95,19.99,0.30,4,107,1
7,55.25,99.99,1.85,4,108,1
8,22.40,44.75,0.73,5,109,1
9,36.90,68.40,1.20,5,110,1


# Killing the service

In [4]:
import json
from azureml.core import Workspace
from azureml.core.webservice import Webservice

# Safety switch: set to True only when you really want to delete the endpoint.
CONFIRM_DELETE = True

# Matches your deployment config in functions/main.py
SERVICE_NAME = "churn-service-v2"
RESOURCE_GROUP = "Papus"
WORKSPACE_NAME = "workspace"

with open("credentials.json", "r") as f:
    creds = json.load(f)

SUBSCRIPTION_ID = creds["subscription_id"]

if not CONFIRM_DELETE:
    raise RuntimeError(
        "Deletion is blocked. Set CONFIRM_DELETE=True to permanently remove the service."
    )

# Connect to Azure ML workspace
ws = Workspace.get(
    name=WORKSPACE_NAME,
    subscription_id=SUBSCRIPTION_ID,
    resource_group=RESOURCE_GROUP,
)

# Delete endpoint/service to stop ACI runtime charges
service = Webservice(name=SERVICE_NAME, workspace=ws)
print(f"Deleting service: {SERVICE_NAME}")
service.delete()
print("Service deleted.")

# Optional hygiene: remove local URI file so future calls fail fast
try:
    with open("uri.json", "w") as f:
        json.dump({"URI": []}, f)
    print("Cleared uri.json.")
except Exception as e:
    print(f"Could not update uri.json: {e}")

Deleting service: churn-service-v2
Running
2026-04-16 15:56:31-06:00 Check and wait for operation (6437ce6b-3796-49be-8ef2-6fc83f674a5f) to finish.
2026-04-16 15:56:32-06:00 Deleting service entity.
Succeeded
Service deleted.
Cleared uri.json.
